# 📊 PROFESSIONAL CORPORATE PPT GENERATOR
### Style: 'Supreme' (Serif Titles, #F3F3F3 Background, Cycle Diagrams)
### Content: AWS Service Proposal

In [ ]:
!pip install python-pptx pydantic -q

In [ ]:
import math
import json
import base64
import io
from typing import List, Optional, Literal, Union, Dict, Any
from pydantic import BaseModel
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.chart.data import CategoryChartData
from pptx.enum.chart import XL_CHART_TYPE, XL_LEGEND_POSITION
from IPython.display import display, HTML

# ==========================================
# 🎨 1. THE DESIGN SYSTEM (CSS FOR PPT)
# ==========================================

class ThemeColors(BaseModel):
    # background: List[int] = [243, 243, 243]  # #F3F3F3 (Light Grey)
    background: List[int] = [255, 255, 255] # Keeping white for clean contrast on diagrams, or change to 243
    primary_text: List[int] = [47, 69, 84]     # #2F4554 (Navy)
    body_text: List[int] = [80, 80, 80]        # Dark Grey
    accent_1: List[int] = [194, 136, 136]      # #C28888 (Muted Red/Clay)
    accent_2: List[int] = [124, 174, 166]      # #7CAEA6 (Teal)
    accent_3: List[int] = [97, 160, 168]       # #61A0A8 (Soft Blue)
    banner_bg: List[int] = [230, 230, 250]     # #E6E6FA (Lavender)
    status_green: List[int] = [0, 153, 51]
    status_blue: List[int] = [0, 112, 192]
    status_red: List[int] = [200, 0, 0]

class ThemeFonts(BaseModel):
    # Typography Rules
    family_serif: str = "Times New Roman"
    family_sans: str = "Arial"
    size_h1: int = 40
    size_h2: int = 18
    size_body: int = 11

# ==========================================
# 📐 2. THE GEOMETRY ENGINE (MATH)
# ==========================================
# This calculates X/Y coordinates to draw perfect circles and process flows

def calculate_circle_points(center_x, center_y, radius, steps):
    points = []
    for i in range(steps):
        # -90 degrees so we start at 12 o'clock
        angle = math.radians((360 / steps) * i - 90)
        x = center_x + radius * math.cos(angle)
        y = center_y + radius * math.sin(angle)
        points.append((x, y))
    return points

# ==========================================
# 📝 3. DATA MODELS
# ==========================================

class SlideCover(BaseModel):
    type: Literal["cover"]
    title: str
    subtitle: str
    author: str

class SlideContent(BaseModel):
    type: Literal["content"]
    title: str
    banner: Optional[str] = None
    text: str

class SlideCycle(BaseModel):
    type: Literal["cycle"]
    title: str
    steps: List[Dict[str, str]] # [{'label': 'Step 1', 'desc': 'Text'}]

class SlideTable(BaseModel):
    type: Literal["table"]
    title: str
    columns: List[str]
    rows: List[List[str]]

class SlideCards(BaseModel):
    type: Literal["cards"]
    title: str
    cards: List[Dict[str, str]]

class PresentationConfig(BaseModel):
    filename: str
    slides: List[Union[SlideCover, SlideContent, SlideCycle, SlideTable, SlideCards]]

print("✅ System Initialized")

In [ ]:
# ==========================================
# 🖌️ 4. THE RENDERER (DRAWING ON CANVAS)
# ==========================================

def get_rgb(c): return RGBColor(c[0], c[1], c[2])

COLORS = ThemeColors()
FONTS = ThemeFonts()

def add_master_elements(slide, title_text):
    # 1. Background Fill
    bg = slide.background
    fill = bg.fill
    fill.solid()
    fill.fore_color.rgb = get_rgb(COLORS.background)

    # 2. H1 Title (Serif, Navy)
    ts = slide.shapes.add_textbox(Inches(0.5), Inches(0.4), Inches(12), Inches(1))
    p = ts.text_frame.paragraphs[0]
    p.text = title_text
    p.font.name = FONTS.family_serif
    p.font.size = Pt(FONTS.size_h1)
    p.font.bold = True
    p.font.color.rgb = get_rgb(COLORS.primary_text)

    # 3. Decorator Line
    line = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0.5), Inches(1.2), Inches(1.5), Inches(0.05))
    line.fill.solid()
    line.fill.fore_color.rgb = get_rgb(COLORS.primary_text)
    line.line.fill.background()

def render_cover(prs, data: SlideCover):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    
    # Background #F3F3F3
    bg = slide.background
    bg.fill.solid()
    bg.fill.fore_color.rgb = get_rgb(COLORS.background)

    # Big Title
    tb = slide.shapes.add_textbox(Inches(1), Inches(3), Inches(11), Inches(2))
    p = tb.text_frame.paragraphs[0]
    p.text = data.title
    p.font.name = FONTS.family_serif
    p.font.size = Pt(60)
    p.font.bold = True
    p.font.color.rgb = get_rgb(COLORS.primary_text)
    p.alignment = PP_ALIGN.LEFT

    # Subtitle
    sb = slide.shapes.add_textbox(Inches(1), Inches(5), Inches(11), Inches(1))
    p = sb.text_frame.paragraphs[0]
    p.text = data.subtitle
    p.font.name = FONTS.family_sans
    p.font.size = Pt(24)
    p.font.color.rgb = get_rgb(COLORS.body_text)

    # Author/Meta
    ab = slide.shapes.add_textbox(Inches(1), Inches(6.5), Inches(11), Inches(0.5))
    p = ab.text_frame.paragraphs[0]
    p.text = data.author
    p.font.size = Pt(14)
    p.font.color.rgb = get_rgb(COLORS.accent_2)

def render_cycle(prs, data: SlideCycle):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    add_master_elements(slide, data.title)

    # Geometry Settings
    center_x = Inches(13.33/2)
    center_y = Inches(7.5/2 + 0.5)
    radius = Inches(2.2)
    steps_count = len(data.steps)
    
    points = calculate_circle_points(13.33/2*914400, (7.5/2+0.5)*914400, 2.2*914400, steps_count)
    
    # Central Icon/Circle
    center_circle = slide.shapes.add_shape(MSO_SHAPE.OVAL, center_x - Inches(1), center_y - Inches(1), Inches(2), Inches(2))
    center_circle.fill.solid()
    center_circle.fill.fore_color.rgb = get_rgb([255, 255, 255])
    center_circle.line.color.rgb = get_rgb(COLORS.accent_3)
    center_circle.line.width = Pt(3)

    # Draw Nodes
    for i, (x_emu, y_emu) in enumerate(points):
        step_data = data.steps[i]
        
        # Connector Line
        line = slide.shapes.add_connector(MSO_SHAPE.STRAIGHT_CONNECTOR_1, center_x, center_y, x_emu, y_emu)
        line.line.color.rgb = get_rgb(COLORS.accent_3)
        line.line.width = Pt(2)
        
        # Node Circle
        node_size = Inches(1.2)
        node = slide.shapes.add_shape(MSO_SHAPE.OVAL, x_emu - node_size/2, y_emu - node_size/2, node_size, node_size)
        node.fill.solid()
        # Rotate Colors
        color_cycle = [COLORS.primary_text, COLORS.accent_1, COLORS.accent_2, COLORS.accent_3]
        node.fill.fore_color.rgb = get_rgb(color_cycle[i % 4])
        node.line.fill.background()
        
        # Label (Step Name)
        p = node.text_frame.paragraphs[0]
        p.text = step_data['label']
        p.font.color.rgb = get_rgb([255, 255, 255])
        p.font.bold = True
        p.font.size = Pt(14)
        
        # Description Text (Outside)
        # Calculate offset for text based on angle to keep it readable
        # Simple implementation: Text box near node
        tb = slide.shapes.add_textbox(x_emu - Inches(1), y_emu + Inches(0.7), Inches(2), Inches(1))
        p = tb.text_frame.paragraphs[0]
        p.text = step_data['desc']
        p.font.size = Pt(10)
        p.alignment = PP_ALIGN.CENTER

def render_table(prs, data: SlideTable):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    add_master_elements(slide, data.title)

    rows = len(data.rows) + 1
    cols = len(data.columns)
    shape = slide.shapes.add_table(rows, cols, Inches(0.5), Inches(2), Inches(12.33), Inches(0.5*rows))
    table = shape.table

    # Headers
    for i, col_name in enumerate(data.columns):
        cell = table.cell(0, i)
        cell.text = col_name
        cell.fill.solid()
        cell.fill.fore_color.rgb = get_rgb(COLORS.primary_text)
        p = cell.text_frame.paragraphs[0]
        p.font.color.rgb = get_rgb([255, 255, 255])
        p.font.bold = True

    # Rows
    for r, row_data in enumerate(data.rows, start=1):
        for c, val in enumerate(row_data):
            cell = table.cell(r, c)
            cell.text = val
            p = cell.text_frame.paragraphs[0]
            p.font.size = Pt(11)
            
            # TRAFFIC LIGHT LOGIC
            val_lower = val.lower()
            if "active" in val_lower or "healthy" in val_lower:
                p.font.color.rgb = get_rgb(COLORS.status_green)
                p.font.bold = True
            elif "critical" in val_lower or "risk" in val_lower:
                p.font.color.rgb = get_rgb(COLORS.status_red)
                p.font.bold = True
            elif "progress" in val_lower:
                p.font.color.rgb = get_rgb(COLORS.status_blue)

def render_content(prs, data: SlideContent):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    add_master_elements(slide, data.title)
    
    cursor_y = 1.8
    if data.banner:
        box = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0.5), Inches(cursor_y), Inches(12.33), Inches(0.6))
        box.fill.solid()
        box.fill.fore_color.rgb = get_rgb(COLORS.banner_bg)
        box.line.color.rgb = get_rgb(COLORS.primary_text)
        box.line.width = Pt(0.5)
        p = box.text_frame.paragraphs[0]
        p.text = data.banner
        p.font.color.rgb = get_rgb(COLORS.primary_text)
        cursor_y += 1.0

    tb = slide.shapes.add_textbox(Inches(0.5), Inches(cursor_y), Inches(12), Inches(4))
    tf = tb.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.text = data.text
    p.font.name = FONTS.family_sans
    p.font.size = Pt(16)

def render_cards(prs, data: SlideCards):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    add_master_elements(slide, data.title)

    count = len(data.cards)
    rows = 2 if count > 3 else 1
    cols = math.ceil(count / rows)
    
    margin_x = 0.5
    start_y = 1.8
    card_w = (13.33 - 1.0 - (0.2 * (cols-1))) / cols
    card_h = 2.5

    for i, card in enumerate(data.cards):
        r = i // cols
        c = i % cols
        x = margin_x + c * (card_w + 0.2)
        y = start_y + r * (card_h + 0.2)
        
        # Card Box
        shape = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(x), Inches(y), Inches(card_w), Inches(card_h))
        shape.fill.solid()
        shape.fill.fore_color.rgb = get_rgb([255, 255, 255])
        shape.line.color.rgb = get_rgb(COLORS.accent_2)
        
        # Card Title
        p = shape.text_frame.paragraphs[0]
        p.text = card['title']
        p.font.bold = True
        p.font.size = Pt(16)
        p.font.color.rgb = get_rgb(COLORS.primary_text)
        
        p = shape.text_frame.add_paragraph()
        p.text = card['desc']
        p.font.size = Pt(11)
        p.font.color.rgb = get_rgb(COLORS.body_text)

print("✅ Renderer Loaded")

In [ ]:
# ==========================================
# 📝 5. THE MASTER DATA (AWS PROPOSAL)
# ==========================================

aws_proposal_json = {
  "filename": "AWS_Corporate_Proposal.pptx",
  "slides": [
    # 1. Cover
    {
      "type": "cover",
      "title": "AWS Cloud Infrastructure",
      "subtitle": "Scalable, Secure, and Cost-Optimized Solutions for Enterprise",
      "author": "Cloud Architecture Team | Dec 2025"
    },
    # 2. Executive Summary (Banner)
    {
      "type": "content",
      "title": "Executive Summary",
      "banner": "Proposal Highlight: Transitioning to AWS will reduce OpEx by 34% within the first 12 months while increasing availability to 99.99%.",
      "text": "Our analysis of the current on-premise infrastructure identifies three critical bottlenecks: limited scalability during peak traffic, high maintenance costs of aging hardware, and security compliance gaps. By migrating to the AWS Well-Architected Framework, we address these immediately."
    },
    # 3. Strategic Cycle (Donut Diagram)
    {
      "type": "cycle",
      "title": "The Modernization Cycle",
      "steps": [
        {"label": "Assess", "desc": "Discovery & TCO Analysis"},
        {"label": "Mobilize", "desc": "Landing Zone Setup"},
        {"label": "Migrate", "desc": "Rehost & Replatform"},
        {"label": "Modernize", "desc": "Serverless & Containers"},
        {"label": "Optimize", "desc": "Cost & Performance Tuning"}
      ]
    },
    # 4. Service Health (Traffic Light Table)
    {
      "type": "table",
      "title": "Current Infrastructure Health Check",
      "columns": ["Service Component", "Region", "Uptime", "Status"],
      "rows": [
        ["Legacy Web Servers", "US-East-1", "98.5%", "Critical Risk"],
        ["Oracle Database", "US-East-1", "99.0%", "At Risk"],
        ["Network Gateway", "Global", "99.9%", "Healthy"],
        ["Backup Systems", "Off-site", "N/A", "In Progress"]
      ]
    },
    # 5. Core Services (Grid Cards)
    {
      "type": "cards",
      "title": "Proposed AWS Core Services",
      "cards": [
        {"title": "Amazon EC2", "desc": "Secure and resizable compute capacity for workloads."},
        {"title": "Amazon S3", "desc": "Object storage built to retrieve any amount of data from anywhere."},
        {"title": "Amazon RDS", "desc": "Managed relational database service for MySQL, PostgreSQL, and SQL Server."},
        {"title": "AWS Lambda", "desc": "Run code without thinking about servers or clusters."},
        {"title": "Amazon VPC", "desc": "Logically isolated virtual network for secure resources."},
        {"title": "CloudWatch", "desc": "Observability of your AWS resources and applications."}
      ]
    },
    # 6. Security Cycle (Donut)
    {
      "type": "cycle",
      "title": "Shared Responsibility Model",
      "steps": [
        {"label": "Identity", "desc": "IAM Roles & MFA"},
        {"label": "Detect", "desc": "GuardDuty & Config"},
        {"label": "Protect", "desc": "WAF & Shield"},
        {"label": "Respond", "desc": "Automated Remediation"}
      ]
    },
    # 7. Cost Analysis (Table)
    {
      "type": "table",
      "title": "Projected Cost Savings (Year 1)",
      "columns": ["Cost Center", "Current On-Prem ($)", "AWS Projected ($)", "Savings Status"],
      "rows": [
        ["Hardware Refresh", "150,000", "0", "Healthy"],
        ["Data Center Power", "45,000", "0", "Healthy"],
        ["Compute Resources", "0", "85,000", "Active Spend"],
        ["Operational Staff", "120,000", "90,000", "Healthy"]
      ]
    },
    # 8. Next Steps (Content)
    {
      "type": "content",
      "title": "Immediate Next Steps",
      "banner": "Approval required by Dec 30 to initiate the Landing Zone creation in Q1.",
      "text": "1. Finalize the Statement of Work (SoW).\n2. Establish the AWS Organization and Billing setup.\n3. Conduct the initial Security Immersion Day for the dev team.\n4. Begin the 'Assess' phase for the legacy database."
    }
  ]
}

print("✅ Data Loaded")

In [ ]:
# ==========================================
# 🚀 6. EXECUTE GENERATOR
# ==========================================

try:
    print("⚙️ Configuring Presentation...")
    config = PresentationConfig(**aws_proposal_json)
    
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)
    
    for slide_data in config.slides:
        if slide_data.type == "cover":
            render_cover(prs, slide_data)
        elif slide_data.type == "content":
            render_content(prs, slide_data)
        elif slide_data.type == "cycle":
            render_cycle(prs, slide_data)
        elif slide_data.type == "table":
            render_table(prs, slide_data)
        elif slide_data.type == "cards":
            render_cards(prs, slide_data)
            
    # Save and Download
    output_file = config.filename
    prs.save(output_file)
    
    with open(output_file, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
        
    print(f"✅ SUCCESS! {output_file} Generated.")
    
    download_btn = f'''
    <a href="data:application/vnd.openxmlformats-officedocument.presentationml.presentation;base64,{b64}" 
       download="{output_file}" 
       style="display: inline-block; padding: 15px 30px; background-color: #2F4554; color: white; 
              text-decoration: none; font-family: serif; font-weight: bold; border-radius: 4px; font-size: 18px;">
       ⬇️ DOWNLOAD CORPORATE DECK
    </a>
    '''
    display(HTML(download_btn))

except Exception as e:
    print(f"❌ ERROR: {e}")